# Phase 5 — MLOps + Deployment
MLflow experiment tracking, the global-popularity fallback artifact for serving, and a local check of the inference path that `app.py` uses.

## Cell 1: Save the global-popularity fallback artifact
`app.py`/`src/inference.py` must not reload `train.parquet` or recompute anything at request time (roadmap: "<10ms, dict lookup, no model call"). Persist the top-10 popularity list once here.

In [1]:
import json
from pathlib import Path

import pandas as pd

import sys
sys.path.insert(0, "..")
from src.baseline import compute_popularity_score

PROCESSED_DIR = Path("../data/processed")

train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
popularity_score = compute_popularity_score(train)
global_popularity_top10 = popularity_score.index[:10].tolist()

with open(PROCESSED_DIR / "global_popularity_top10.json", "w") as f:
    json.dump(global_popularity_top10, f)

print("global_popularity_top10:", global_popularity_top10)
print("saved global_popularity_top10.json")

global_popularity_top10: [461686, 5411, 187946, 257040, 309778, 370653, 7943, 298009, 369447, 48030]
saved global_popularity_top10.json


## Cell 2: MLflow — two tracked runs
Real params (already tuned/fixed in Phases 2-3) and real test-set metrics (Phase 4, `baseline_metrics_test.json` / `final_metrics_test.json`) — nothing here is re-derived or approximated.

In [2]:
import pickle
import subprocess
import time

import mlflow

with open(PROCESSED_DIR / "baseline_metrics_test.json") as f:
    baseline_metrics = json.load(f)
with open(PROCESSED_DIR / "final_metrics_test.json") as f:
    final_metrics = json.load(f)
with open(PROCESSED_DIR / "id_mappings.pkl", "rb") as f:
    id_mappings = pickle.load(f)
best_alpha = id_mappings["best_alpha"]

try:
    dataset_version = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd="..").decode().strip()
except Exception:
    dataset_version = "unknown"

mlflow.set_tracking_uri(f"file:{(Path('..') / 'mlruns').resolve()}")
mlflow.set_experiment("retailrocket-recommender")

with mlflow.start_run(run_name="popularity_baseline"):
    mlflow.log_param("dataset", "retailrocket")
    mlflow.log_param("model_type", "popularity")
    mlflow.log_param("K", 10)
    for k, v in baseline_metrics.items():
        mlflow.log_metric(k, v)
print("logged run: popularity_baseline")

# Re-time the XGBoost fit for an honest training_time_sec metric (cheap, ~6s) rather than
# retrofitting a timing-export step into the already-committed Phase 3 notebook.
from src.ranker import build_training_set, train_ranker

candidates = pd.read_parquet(PROCESSED_DIR / "candidates.parquet")
val = pd.read_parquet(PROCESSED_DIR / "val.parquet")
user_features = pd.read_parquet(PROCESSED_DIR / "user_features.parquet")
item_features = pd.read_parquet(PROCESSED_DIR / "item_features.parquet")
user_item_features = pd.read_parquet(PROCESSED_DIR / "user_item_features.parquet")

val_target_events = val[val["event"].isin(["addtocart", "transaction"])]
training_set = build_training_set(candidates, user_features, item_features, user_item_features, positive_pairs=val_target_events)

t0 = time.time()
_ = train_ranker(training_set)
training_time_sec = time.time() - t0

with mlflow.start_run(run_name="candidate_gen_xgboost_ranking"):
    mlflow.log_param("dataset", "retailrocket")
    mlflow.log_param("model_type", "als_itemcf_candidates+xgboost_ranker")
    mlflow.log_param("als_factors", 50)
    mlflow.log_param("als_iterations", 20)
    mlflow.log_param("alpha", best_alpha)
    mlflow.log_param("itemcf_top_n", 20)
    mlflow.log_param("candidate_pool_size", 50)
    mlflow.log_param("xgb_objective", "binary:logistic")
    mlflow.log_param("xgb_max_depth", 5)
    mlflow.log_param("xgb_n_estimators", 200)
    mlflow.log_param("xgb_learning_rate", 0.1)
    mlflow.log_param("feature_version", "v1")
    mlflow.log_param("dataset_version", dataset_version)
    mlflow.log_param("K", 10)
    mlflow.log_metric("training_time_sec", training_time_sec)
    for k, v in final_metrics.items():
        mlflow.log_metric(k, v)
    mlflow.log_artifact(str(PROCESSED_DIR / "als_model.pkl"))
    mlflow.log_artifact(str(PROCESSED_DIR / "itemcf_matrix.pkl"))
    mlflow.log_artifact(str(PROCESSED_DIR / "xgboost_ranker.pkl"))
print(f"logged run: candidate_gen_xgboost_ranking (training_time_sec={training_time_sec:.1f}s)")

logged run: popularity_baseline


logged run: candidate_gen_xgboost_ranking (training_time_sec=6.1s)


## Cell 3: Verify the inference path `app.py` uses
Known warm user -> personalized; unknown user -> popularity fallback with exactly K items. No model objects loaded, just the two artifacts `src/inference.py` reads.

In [3]:
from src.inference import load_serving_artifacts, recommend

precomputed_final_recs, global_popularity = load_serving_artifacts()
print(f"precomputed personalized users: {len(precomputed_final_recs):,}")
print(f"global_popularity fallback: {global_popularity}")

known_uid = next(iter(precomputed_final_recs))
recs, source = recommend(known_uid, precomputed_final_recs, global_popularity, k=10)
print(f"\nknown warm user {known_uid}: source={source}, recs={recs}")

unknown_uid = 999999999
recs, source = recommend(unknown_uid, precomputed_final_recs, global_popularity, k=10)
print(f"\nunknown user {unknown_uid}: source={source}, recs={recs}")
assert len(recs) == 10 and source == "popularity_fallback"
print("\nfallback behavior verified")

precomputed personalized users: 11,587
global_popularity fallback: [461686, 5411, 187946, 257040, 309778, 370653, 7943, 298009, 369447, 48030]

known warm user 155: source=personalized, recs=[134620, 123027, 107229, 304291, 220749, 442395, 434684, 191188, 56209, 216804]

unknown user 999999999: source=popularity_fallback, recs=[461686, 5411, 187946, 257040, 309778, 370653, 7943, 298009, 369447, 48030]

fallback behavior verified
